# Notebook 02: Concurrencia, Asincronía y asyncio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/16_computo/code/02_concurrencia_asyncio.ipynb)

**Módulo 16 — Clase 2**

Este notebook acompaña los archivos `03_concurrencia_y_asincronia.md` y `04a_asyncio_fundamentos.md`.

Secciones **** se trabajan durante la sesión.  
Secciones **** se completan después.

---

In [9]:
import asyncio
import time
import threading
import os
import sys

print(f'Python {sys.version}')
print(f'asyncio version: {asyncio.__version__ if hasattr(asyncio, "__version__") else "built-in"}')

# Jupyter ya tiene un event loop corriendo — podemos usar await directamente en las celdas
# Si usas un script .py, necesitas asyncio.run(main())

Python 3.12.0 (main, Mar 17 2026, 23:44:31) [Clang 17.0.0 (clang-1700.6.4.2)]
asyncio version: built-in


## Sección 1: await secuencial vs asyncio.gather — la diferencia central

**Qué vamos a ver:** la diferencia entre M2 (await secuencial) y M4 (gather) es literalmente una línea de código. Los tiempos medidos hacen que la diferencia sea imposible de ignorar.

**Predicción del modelo:**
- M2 (await en secuencia): `T_total = N × T_tarea` — cada usuario espera a que el anterior termine
- M4 (gather): `T_total ≈ T_tarea_más_lenta` — todos los usuarios esperan *al mismo tiempo*

Con N=5 tareas de 1.0s cada una:
- M2 debería tardar: **5.0s**
- M4 debería tardar: **~1.0s**

Corre las celdas y verifica. ¿Coincide con la predicción? ¿El speedup es exactamente 5×, o hay overhead?

> Referencia: `04a_asyncio_fundamentos.md` — sección "asyncio.gather — M4 en una línea"

In [10]:
# Tarea simulada con I/O-bound: espera τ segundos
async def tarea_io(nombre: str, duracion: float) -> str:
    # exec(τᵢ): inicializar
    inicio = time.perf_counter()
    # wait(τᵢ): simula I/O (llamada a API, lectura de BD, etc.)
    await asyncio.sleep(duracion)
    # exec(τᵢ): procesar resultado
    elapsed = time.perf_counter() - inicio
    return f'{nombre}: {elapsed:.2f}s'

DURACION = 1.0  # cada tarea tarda 1s de I/O
N_TAREAS = 5

# --- M2: await secuencial (esperas NO explotadas) ---
t0 = time.perf_counter()
resultados_m2 = []
for i in range(N_TAREAS):
    r = await tarea_io(f'τ{i+1}', DURACION)
    resultados_m2.append(r)
t_m2 = time.perf_counter() - t0

print(f'=== M2: await secuencial ===')
for r in resultados_m2:
    print(f'  {r}')
print(f'Tiempo total M2: {t_m2:.2f}s  (esperado: {N_TAREAS * DURACION:.1f}s = N×T)')
print()

=== M2: await secuencial ===
  τ1: 1.00s
  τ2: 1.00s
  τ3: 1.00s
  τ4: 1.00s
  τ5: 1.00s
Tiempo total M2: 5.00s  (esperado: 5.0s = N×T)



In [11]:
# --- M4: asyncio.gather (esperas SÍ explotadas) ---
t0 = time.perf_counter()
resultados_m4 = await asyncio.gather(
    *[tarea_io(f'τ{i+1}', DURACION) for i in range(N_TAREAS)]
)
t_m4 = time.perf_counter() - t0

print(f'=== M4: asyncio.gather ===')
for r in resultados_m4:
    print(f'  {r}')
print(f'Tiempo total M4: {t_m4:.2f}s  (esperado: ~{DURACION:.1f}s = T_max)')
print()
print(f'Speedup M4/M2: {t_m2/t_m4:.1f}x')
print()
print(f'Conclusión: gather explota las esperas — exec(τⱼ) ∩ wait(τᵢ) ≠ ∅')
print(f'Las {N_TAREAS} tareas de {DURACION}s corren en ~{DURACION}s en lugar de {N_TAREAS*DURACION}s')

=== M4: asyncio.gather ===
  τ1: 1.00s
  τ2: 1.00s
  τ3: 1.00s
  τ4: 1.00s
  τ5: 1.00s
Tiempo total M4: 1.00s  (esperado: ~1.0s = T_max)

Speedup M4/M2: 5.0x

Conclusión: gather explota las esperas — exec(τⱼ) ∩ wait(τᵢ) ≠ ∅
Las 5 tareas de 1.0s corren en ~1.0s en lugar de 5.0s


## Sección 2: Traza del event loop con asyncio debug mode

**Qué vamos a ver:** asyncio tiene un modo de depuración que emite advertencias cuando el event loop se bloquea más tiempo del esperado. Es la herramienta para diagnosticar el anti-patrón de `time.sleep` dentro de funciones `async`.

**El problema a demostrar:**
```
time.sleep(0.3)       ← bloquea el hilo del OS durante 300ms
                         → el event loop no puede ejecutar NINGUNA otra coroutine
                         → gather con 3 tareas de 0.3s tarda 0.9s (secuencial)

await asyncio.sleep(0.3)  ← registra un callback y cede el control
                              → el event loop puede ejecutar otras coroutines
                              → gather con 3 tareas de 0.3s tarda ~0.3s (M4)
```

**Predicción:**
- `gather` con `asyncio.sleep(0.3)`: debería tardar **~0.3s**
- `gather` con `time.sleep(0.3)`: debería tardar **~0.9s** (sin mejora — M1)

El modo debug (`loop.set_debug(True)`) detectará el bloqueo y emitirá una advertencia en el segundo caso. Observa el output completo.

> Referencia: `04a_asyncio_fundamentos.md` — sección "time.sleep vs asyncio.sleep"

In [12]:
import asyncio
import time

# Habilitamos debug mode para ver bloqueos
loop = asyncio.get_event_loop()
loop.set_debug(True)

# Un umbral bajo para detectar bloqueos rápidamente
# (normalmente el umbral es 100ms)
loop.slow_callback_duration = 0.05  # 50ms

# Tarea bien escrita: libera el event loop
async def tarea_correcta(nombre: str):
    print(f'  {nombre}: inicio')
    await asyncio.sleep(0.3)   # wait(τ) — event loop libre
    print(f'  {nombre}: fin')

# Tarea mal escrita: BLOQUEA el event loop
async def tarea_bloqueante(nombre: str):
    print(f'  {nombre}: inicio')
    time.sleep(0.3)            # ← bloquea el hilo del OS entero
    print(f'  {nombre}: fin')

# ¿Qué diferencia ves en la salida?
print('=== gather con tareas CORRECTAS (asyncio.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_correcta('A'), tarea_correcta('B'), tarea_correcta('C'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.3s)\n')

print('=== gather con tareas BLOQUEANTES (time.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_bloqueante('X'), tarea_bloqueante('Y'), tarea_bloqueante('Z'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.9s — sin mejora)')
print()
print('Observa: con time.sleep, gather NO ayuda.')
print('time.sleep bloquea el event loop → ninguna otra coroutine puede avanzar.')

=== gather con tareas CORRECTAS (asyncio.sleep) ===
  A: inicio
  B: inicio
  C: inicio
  A: fin
  B: fin
  C: fin
Tiempo: 0.30s  (esperado: ~0.3s)

=== gather con tareas BLOQUEANTES (time.sleep) ===
  X: inicio
  X: fin
  Y: inicio
  Y: fin
  Z: inicio
  Z: fin
Tiempo: 0.92s  (esperado: ~0.9s — sin mejora)

Observa: con time.sleep, gather NO ayuda.
time.sleep bloquea el event loop → ninguna otra coroutine puede avanzar.


In [13]:
# Desactivar debug mode para el resto del notebook
loop.set_debug(False)

---

## Sección 3: Implementar M2 y M3 — por qué NO mejoran

**TAREA — implementación guiada**

Esta sección te pide implementar dos modelos *incorrectos* para CPU-bound e ineficientes para sus casos de uso, y medir por qué fallan la promesa de concurrencia/paralelismo.

**TAREA 3.1 — M2: async con await secuencial**

Ya viste M2 en la Sección 1. Ahora explícalo formalmente:
- ¿Qué condición de M4 (`exec(τⱼ) ∩ wait(τᵢ) ≠ ∅`) falla en M2?
- ¿En qué se diferencia `await fn1(); await fn2()` de `asyncio.gather(fn1(), fn2())`?

Escribe la respuesta como comentario en la celda antes de correr el código.

**TAREA 3.2 — M3: threading CPU-bound**

El GIL (Global Interpreter Lock) garantiza que solo un hilo ejecuta bytecode Python a la vez. Para tareas CPU-bound (`wait(τᵢ) = ∅`), el GIL nunca se libera — el threading no puede producir paralelismo real.

**Predicción:** con N=4 hilos en una tarea CPU-bound, el speedup esperado es **≈1×** (sin mejora). Puede incluso ser < 1× por el overhead de sincronización del GIL.

Implementa el threading y mide. ¿Coincide con la predicción?

In [15]:
import time
import threading

# TAREA 3.1 — M2: async con await secuencial (ya visto en Sección 1)
# Pregunta: ¿por qué M2 es idéntico a M1 en términos de tiempo?
# Responde con la definición formal: ¿qué condición de M4 falta en M2?
#
# RESPUESTA:
# En M2 falla la condición fundamental de la concurrencia: que la ejecución de una
# tarea se solape con el tiempo de espera de otra (exec de J intersecta wait de I
# no debe ser nulo).
# Al usar 'await fn1()' seguido de 'await fn2()', nunca inscribimos la tarea 2 
# en el Event Loop hasta que la tarea 1 haya terminado por completo. Por lo tanto, 
# el tiempo de espera de la primera no se solapa con la ejecución (ni la espera) de la segunda.
# 
# Diferencia clave:
# 'await fn1(); await fn2()' = "Pausa aquí hasta que fn1 acabe, luego arranca fn2".
# 'asyncio.gather(fn1(), fn2())' = "Arranca ambas al mismo tiempo; pausa aquí hasta que ambas acaben".

# TAREA 3.2 — M3: threading CPU-bound
# Implementa N tareas CPU-bound con threading y mide vs secuencial.
# ¿Coincide con la predicción del GIL (sin speedup, posible slowdown)?

def tarea_cpu_bound(n: int) -> int:
    """Tarea CPU-bound pura: no tiene tiempos de espera (wait = nulo)"""
    return sum(range(n))

N_CPU = 30_000_000
N_HILOS = 4

# --- Secuencial (M1) ---
t0 = time.perf_counter()
for _ in range(N_HILOS):
    tarea_cpu_bound(N_CPU)
t_secuencial = time.perf_counter() - t0

# --- Threading M3 ---
t0_hilos = time.perf_counter()

# 1. Creamos los 4 hilos apuntando a la misma tarea
hilos = [threading.Thread(target=tarea_cpu_bound, args=(N_CPU,)) for _ in range(N_HILOS)]

# 2. Iniciamos todos los hilos
for h in hilos:
    h.start()

# 3. Esperamos a que todos terminen (join bloquea hasta que el hilo finalice)
for h in hilos:
    h.join()

t_threading = time.perf_counter() - t0_hilos

# --- Resultados ---
speedup = t_secuencial / t_threading

print(f'M1 secuencial: {t_secuencial:.3f}s')
print(f'M3 threading:  {t_threading:.3f}s')
print(f'Speedup:       {speedup:.2f}x')
print()
print('¿Qué dice el resultado sobre M3 + GIL en Python?')
print('El resultado confirma que para tareas CPU-bound, threading NO produce paralelismo.')
print('El GIL obliga a los hilos a ejecutarse uno por uno, y el overhead del sistema')
print('operativo cambiando entre hilos suele hacer que M3 sea más lento que M1.')

M1 secuencial: 0.914s
M3 threading:  0.907s
Speedup:       1.01x

¿Qué dice el resultado sobre M3 + GIL en Python?
El resultado confirma que para tareas CPU-bound, threading NO produce paralelismo.
El GIL obliga a los hilos a ejecutarse uno por uno, y el overhead del sistema
operativo cambiando entre hilos suele hacer que M3 sea más lento que M1.


## Sección 4: Race condition reproducible + fix con Lock

**TAREA — reproducir y corregir**

Las condiciones de carrera (race conditions) son el bug clásico de la concurrencia con memoria compartida. Ocurren cuando múltiples hilos leen y escriben la misma variable sin coordinación.

**Por qué ocurre:**
La operación `contador += 1` parece atómica pero en realidad son 3 pasos:
```
LOAD  contador      → lee el valor actual al registro de la CPU
ADD   registro, 1   → incrementa en 1
STORE registro → contador  → escribe de vuelta
```
Si dos hilos ejecutan esto concurrentemente, el segundo puede leer el valor *antes* de que el primero escriba su resultado — se pierde un incremento.

**Predicción:**
- Sin lock: `contador` final < `N_INCREMENTOS × N_HILOS` (incrementos perdidos, no determinístico)
- Con `threading.Lock`: `contador` final = `N_INCREMENTOS × N_HILOS` siempre

Corre varias veces sin lock. ¿El resultado es siempre distinto? ¿Siempre menor que el esperado?

> Nota: el GIL de Python reduce (no elimina) las race conditions. Este ejemplo las reproduce porque el increment no es atómico incluso con el GIL.

In [18]:
import threading
import time

# TAREA 4.1 — Reproduce la race condition
# Reducimos los incrementos para no hacer lenta la ejecución con el sleep
N_INCREMENTOS = 100_000
N_HILOS_RACE = 4

# Sin lock — resultado no determinista
contador_sin_lock = [0]

def incrementar_sin_lock():
    for _ in range(N_INCREMENTOS):
        # EXPLICACIÓN DEL CAMBIO FORZADO:
        # Los procesadores modernos son tan rápidos que pueden terminar las
        # iteraciones antes de que el GIL haga el cambio de hilo (context switch).
        # Al separar la operación y meter un sleep microscópico, obligamos al
        # intérprete a soltar el hilo exactamente a la mitad de la operación.
        # Esto garantiza que múltiples hilos lean la misma base desactualizada.
        
        actual = contador_sin_lock[0]   # 1. LOAD: Leemos el valor
        time.sleep(0.000001)            # 2. Forzamos el cambio de hilo
        contador_sin_lock[0] = actual + 1 # 3. ADD y STORE: Guardamos pisando otros datos

hilos = [threading.Thread(target=incrementar_sin_lock) for _ in range(N_HILOS_RACE)]
for h in hilos: h.start()
for h in hilos: h.join()

esperado = N_INCREMENTOS * N_HILOS_RACE
print('=== TAREA 4.1: Sin Lock (Race Condition Forzada) ===')
print(f'Sin lock  — esperado: {esperado:,}, obtenido: {contador_sin_lock[0]:,}')
print(f'Diferencia: {esperado - contador_sin_lock[0]:,} incrementos perdidos')
print()

# TAREA 4.2 — Fix con Lock
candado = threading.Lock()
contador_con_lock = [0]

def incrementar_con_lock():
    for _ in range(N_INCREMENTOS):
        # El bloque 'with' asegura que solo un hilo a la vez pueda entrar aquí.
        # Aunque el sistema quiera cambiar de hilo por el tiempo, los demás
        # chocarán con la "puerta cerrada" y tendrán que esperar.
        with candado:
            actual = contador_con_lock[0]
            # Mantenemos el sleep para demostrar que el Lock resiste hasta
            # las interrupciones más agresivas
            time.sleep(0.000001)
            contador_con_lock[0] = actual + 1

hilos_lock = [threading.Thread(target=incrementar_con_lock) for _ in range(N_HILOS_RACE)]
for h in hilos_lock: h.start()
for h in hilos_lock: h.join()

print('=== TAREA 4.2: Fix con Lock ===')
print(f'Con lock  — esperado: {esperado:,}, obtenido: {contador_con_lock[0]:,}')

if esperado == contador_con_lock[0]:
    print('¡Exito! El lock serializo las modificaciones y evito la race condition.')

=== TAREA 4.1: Sin Lock (Race Condition Forzada) ===
Sin lock  — esperado: 400,000, obtenido: 100,002
Diferencia: 299,998 incrementos perdidos

=== TAREA 4.2: Fix con Lock ===
Con lock  — esperado: 400,000, obtenido: 400,000
¡Exito! El lock serializo las modificaciones y evito la race condition.


## Sección 5: Chatbot v2 con asyncio — N usuarios concurrentes

**TAREA — implementación completa**

Implementa el servidor chatbot v2 del Escenario A: LLM como API remota, I/O-bound. Compara la versión secuencial (v1) con la concurrente (v2).

**Arquitectura del chatbot v2:**
```
N usuarios simultáneos
        │
asyncio.gather(handle_request(0), handle_request(1), ..., handle_request(N-1))
        │
  ┌─────┴──────────────────────────────────────────┐
  │  Event loop (1 hilo)                           │
  │                                                │
  │  τ_u0 exec → await BD(50ms) → await LLM(1.5s) │
  │    τ_u1 exec → await BD(50ms) → await LLM(1.5s)│
  │      τ_u2 exec → await BD(50ms) → await LLM...│
  └──────────────────┬─────────────────────────────┘
                     │ (simultáneamente)
            [BD asyncpg]  [LLM API aiohttp]
```

**Predicciones:**
- v1 (secuencial) con N=10: `T_total ≈ 10 × 1.55s = 15.5s`
- v2 (gather) con N=10: `T_total ≈ 1.55s`
- Latencia del usuario 10 en v2: **similar a la del usuario 1** (todos esperan en paralelo)

Implementa `servidor_v1` y `servidor_v2`, mide con N=10, y responde:
1. ¿La latencia es uniforme entre usuarios en v2? ¿Por qué?
2. ¿Qué pasaría si uno de los usuarios tuviera `time.sleep` en su handler?

> Referencia: `04a_asyncio_fundamentos.md` — sección "Chatbot v2"

In [22]:
import asyncio
import time
import random
from statistics import mean

# Simula una consulta a base de datos: operación I/O-bound corta
async def consultar_bd(user_id):
    await asyncio.sleep(0.05)
    return [f"historial del usuario {user_id}"]

# Simula una llamada a un LLM: operación I/O-bound más lenta
async def llamar_llm(historial):
    await asyncio.sleep(random.uniform(1.0, 2.0))
    return f"respuesta para {historial[-1]}"

# Procesa una petición completa de un usuario
async def handle_request(user_id, t_llegada):
    t_inicio = time.perf_counter()

    historial = await consultar_bd(user_id)
    respuesta = await llamar_llm(historial)

    t_fin = time.perf_counter()

    return {
        "user": user_id,
        "respuesta": respuesta,
        "latencia_servicio": t_fin - t_inicio,
        "latencia_desde_llegada": t_fin - t_llegada,
    }

# Versión 1: atiende usuarios uno por uno
async def servidor_v1(n_usuarios):
    t_llegada = time.perf_counter()
    resultados = []

    for user_id in range(n_usuarios):
        resultado = await handle_request(user_id, t_llegada)
        resultados.append(resultado)

    return resultados

# Versión 2: atiende usuarios concurrentemente
async def servidor_v2(n_usuarios):
    t_llegada = time.perf_counter()

    tareas = [
        handle_request(user_id, t_llegada)
        for user_id in range(n_usuarios)
    ]

    return await asyncio.gather(*tareas)

# Imprime las métricas principales
def imprimir_metricas(nombre, resultados, tiempo_total):
    latencias_servicio = [r["latencia_servicio"] for r in resultados]
    latencias_llegada = [r["latencia_desde_llegada"] for r in resultados]

    print(f"=== {nombre} ===")
    print(f"Tiempo total: {tiempo_total:.2f}s")
    print(f"Latencia media de servicio: {mean(latencias_servicio):.2f}s")
    print(f"Latencia media desde llegada: {mean(latencias_llegada):.2f}s")
    print(f"Latencia máxima desde llegada: {max(latencias_llegada):.2f}s")
    print(f"Latencia del último usuario: {resultados[-1]['latencia_desde_llegada']:.2f}s")
    print()

# -----------------------------
# Comparación experimental
# -----------------------------

N = 10
SEED = 42

print(f"Comparando v1 y v2 con {N} usuarios...\n")

# Servidor v1
random.seed(SEED)
t0 = time.perf_counter()
res_v1 = await servidor_v1(N)
t_total_v1 = time.perf_counter() - t0

imprimir_metricas("Servidor v1 — Secuencial", res_v1, t_total_v1)

# Servidor v2
random.seed(SEED)
t0 = time.perf_counter()
res_v2 = await servidor_v2(N)
t_total_v2 = time.perf_counter() - t0

imprimir_metricas("Servidor v2 — Concurrente", res_v2, t_total_v2)

# Speedup
speedup = t_total_v1 / t_total_v2
print("=== Comparación ===")
print(f"Speedup de v2 frente a v1: {speedup:.1f}x")
print()

# -----------------------------
# Respuestas del ejercicio
# -----------------------------

print("=== Respuestas ===")
print()

print("1. ¿La latencia es uniforme entre usuarios en v2?")
print(
    "No completamente. En v2 los usuarios se atienden concurrentemente, "
    "por lo que ninguno espera a que terminen todos los anteriores. "
    "Sin embargo, las latencias no son idénticas porque la llamada al LLM "
    "tiene una duración aleatoria entre 1 y 2 segundos."
)
print()

print("2. ¿Qué pasa si uno de los usuarios usa time.sleep en su handler?")
print(
    "time.sleep bloquearía el event loop completo. "
    "Eso haría que todos los demás usuarios también se detuvieran, "
    "aunque sus funciones usen await correctamente. "
    "Por eso en código async debe usarse await asyncio.sleep."
)

Comparando v1 y v2 con 10 usuarios...

=== Servidor v1 — Secuencial ===
Tiempo total: 14.53s
Latencia media de servicio: 1.45s
Latencia media desde llegada: 8.08s
Latencia máxima desde llegada: 14.53s
Latencia del último usuario: 14.53s

=== Servidor v2 — Concurrente ===
Tiempo total: 1.95s
Latencia media de servicio: 1.45s
Latencia media desde llegada: 1.45s
Latencia máxima desde llegada: 1.94s
Latencia del último usuario: 1.08s

=== Comparación ===
Speedup de v2 frente a v1: 7.5x

=== Respuestas ===

1. ¿La latencia es uniforme entre usuarios en v2?
No completamente. En v2 los usuarios se atienden concurrentemente, por lo que ninguno espera a que terminen todos los anteriores. Sin embargo, las latencias no son idénticas porque la llamada al LLM tiene una duración aleatoria entre 1 y 2 segundos.

2. ¿Qué pasa si uno de los usuarios usa time.sleep en su handler?
time.sleep bloquearía el event loop completo. Eso haría que todos los demás usuarios también se detuvieran, aunque sus funcio